# 4. Excitación fisiológica (velocidad y pupila) y figuras
Sube `data_train.zip` + `velocidad.zip` (para velocidad), `data_pupil.zip` (para pupila) y los `.json` de las notebooks 2 y 3 (para las figuras).

In [ ]:
!pip -q install numba scipy pandas matplotlib numpy

In [ ]:
%%writefile models.py
"""Trial-level models of lick / withhold behaviour in a change-detection task.

All models share one observation model and the same nuisance terms; they differ ONLY in how
the value difference and the inverse temperature evolve:

    logit P(lick_t) = beta_t * (V_lick - V_withhold) + b + w * sat_t
    sat_t           = cumulative reward before trial t, divided by 100  (satiety / motivation proxy)

bias      no learning: logit = b + w*sat
q         TD learning, fixed alpha, fixed beta                       (= DMA with the gate switched off)
ph        Pearce-Hall style: surprise gates the LEARNING RATE
dma       surprise gates the INVERSE TEMPERATURE through a bAP/HCN conductance g_t
          (the discrete DMA mapping of the reference implementation); alpha, beta_base, lambda, tau_hcn free
dma_const same equations with the constants published for DMA, only (b, w) fitted

Teacher forcing: the observed action drives learning; we score its probability.
Reward = 1 only for a lick on a go trial (as in the task); delta uses the chosen action's value
exactly like the reference implementation.
"""
from __future__ import annotations

import math

import numpy as np

try:  # numba is optional; without it everything still runs, only much slower
    from numba import njit
except Exception:  # pragma: no cover
    def njit(*a, **k):
        if len(a) == 1 and callable(a[0]):
            return a[0]
        return lambda f: f

# Published DMA constants (reference implementation)
ALPHA_C, TAU_C, GMAX, VMAX, LAMBDA_C, BETA_C = 0.15, 45.0, 1.0, 1.0, 0.8, 5.5

MODELS = ["bias", "q", "ph", "dma", "dma_const", "ck", "epi", "hmm", "hmmck", "qck", "dmack"]
MID = {m: i for i, m in enumerate(MODELS)}

# name, bounds, starting points
SPECS = {
    "bias": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[0.0, 0.0, 3.0]]),
    "q": (["alpha", "beta", "b", "w", "s"], [(0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
          [[0.15, 4.0, -4.0, 0.0, 3.0], [0.4, 8.0, -7.0, -1.0, 4.0]]),
    "ph": (["alpha0", "kappa", "eta", "beta", "b", "w", "s"],
           [(0.01, 0.9), (0.0, 5.0), (0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
           [[0.15, 1.0, 0.2, 4.0, -4.0, 0.0, 3.0], [0.3, 0.3, 0.5, 8.0, -7.0, -1.0, 4.0]]),
    "dma": (["alpha", "beta_base", "lam", "tau_hcn", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 5.5, 0.8, 45.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, -7.0, -1.0, 4.0]]),
    "dma_const": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[-5.0, 0.0, 3.0], [-7.0, -1.0, 4.0]]),
    # episodic-retrieval models (adapted from the reference implementation's episodic memory):
    # ck  : recency-weighted retrieval of past choices made under the same cue (a plain choice kernel; control)
    # epi : ck plus retrieval of "problem" engrams (a lick that was not rewarded) under the same cue
    "ck": (["rho", "g1", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
           [[0.9, 1.0, -4.0, 0.0, 3.0], [0.6, 2.0, -3.0, 0.0, 3.0]]),
    # latent engagement: two-state sticky HMM, state-specific bias b_k and cue gain s_k (cf. Ashwood et al. 2022)
    "hmm": (["p11", "p22", "b1", "b2", "s1", "s2", "w"],
            [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8)],
            [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0]]),
    "hmmck": (["p11", "p22", "b1", "b2", "s1", "s2", "w", "rho", "g1"],
              [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8), (0.3, 0.999), (-12, 12)],
              [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0, 0.85, 1.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0, 0.6, 2.0]]),
    # value learning PLUS choice retrieval (perseveration): TD or DMA gate on the value term, CK term added to the logit
    "qck": (["alpha", "beta", "rho", "g1", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 4.0, 0.85, 1.0, -4.0, 0.0, 3.0], [0.4, 8.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "dmack": (["alpha", "beta_base", "lam", "tau_hcn", "rho", "g1", "b", "w", "s"],
              [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
              [[0.15, 5.5, 0.8, 45.0, 0.85, 1.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "epi": (["rho", "g1", "g2", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.9, 1.0, -1.0, -4.0, 0.0, 3.0], [0.6, 2.0, 0.0, -3.0, 0.0, 3.0]]),
}


@njit(cache=True)
def _core(mid, P, action, reward, sat, u, rewarded, cue, state, simulate, ll, beta_tr, delta_tr, g_tr):
    n = action.shape[0]
    v0 = 0.33
    V = np.full((2, 2), v0)   # V[state, action]
    g = GMAX
    sm = 0.0
    cum = 0.0
    num_a = np.zeros(2); num_p = np.zeros(2); den = np.zeros(2)   # per-cue episodic traces (models ck, epi)
    for t in range(n):
        s = cum / 100.0 if simulate else sat[t]
        st = state[t]
        V0 = V[st, 0]
        V1 = V[st, 1]
        # ---- policy -------------------------------------------------------------
        if mid == 0:
            beta = 1.0
            z = P[0] + P[1] * s
        elif mid == 1:
            beta = P[1]
            z = beta * (V1 - V0) + P[2] + P[3] * s
        elif mid == 2:
            beta = P[3]
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 3:
            beta = P[1] * g / GMAX
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 4:
            beta = BETA_C * g / GMAX
            z = beta * (V1 - V0) + P[0] + P[1] * s
        elif mid == 9:
            beta = P[1]
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[3] * ma + P[4] + P[5] * s
        elif mid == 10:
            beta = P[1] * g / GMAX
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[5] * ma + P[6] + P[7] * s
        elif mid == 5:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] + P[3] * s
        else:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            mp = num_p[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] * mp + P[3] + P[4] * s
        z += P[P.shape[0] - 1] * cue[t]
        if z > 30.0:
            z = 30.0
        elif z < -30.0:
            z = -30.0
        pl = 1.0 / (1.0 + math.exp(-z))
        if simulate:
            a = 1 if u[t] < pl else 0
            r = 1 if (a == 1 and rewarded[t] == 1) else 0
            action[t] = a
            reward[t] = r
            sat[t] = s
        else:
            a = action[t]
            r = reward[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12:
            pa = 1e-12
        ll[t] = math.log(pa)
        beta_tr[t] = beta
        g_tr[t] = g
        cum += r
        if mid >= 9:
            cc = cue[t]
            rho = P[2] if mid == 9 else P[4]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0
        if mid == 0:
            delta_tr[t] = 0.0
            continue
        if mid == 5 or mid == 6:  # store this trial as an engram of its cue and decay the older ones
            cc = cue[t]
            rho = P[0]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            num_p[cc] = rho * num_p[cc] + (1.0 if (a == 1 and r == 0) else 0.0)
            den[cc] = rho * den[cc] + 1.0
            delta_tr[t] = 0.0
            continue
        # ---- learning -----------------------------------------------------------
        Va = V1 if a == 1 else V0
        delta = r - Va
        delta_tr[t] = delta
        if mid == 1 or mid == 9 or mid == 10:
            a_t = P[0]
        elif mid == 2:
            a_t = P[0] * (1.0 + P[1] * sm)
            if a_t > 0.95:
                a_t = 0.95
            sm = (1.0 - P[2]) * sm + P[2] * abs(delta)
        elif mid == 3:
            a_t = P[0]
        else:
            a_t = ALPHA_C
        Vn = Va + a_t * delta
        if mid == 3 or mid == 4 or mid == 10:  # DMA clips values to [0,1]
            Vn = min(1.0, max(0.0, Vn))
        V[st, a] = Vn
        if mid == 3 or mid == 4 or mid == 10:
            lam = P[2] if (mid == 3 or mid == 10) else LAMBDA_C
            tau = P[3] if (mid == 3 or mid == 10) else TAU_C
            v_ap = VMAX * (1.0 - lam * abs(delta))
            phi = 1.0 / (1.0 + math.exp(-(v_ap - 0.45) / 0.12))
            g = g + (1.0 / tau) * (GMAX * phi - g)
            if g < 0.01:
                g = 0.01
            elif g > GMAX:
                g = GMAX


@njit(cache=True)
def _hmm(P, action, sat, cue, use_ck, ll):
    n = action.shape[0]
    p11 = P[0]; p22 = P[1]
    pi1 = (1.0 - p22) / (2.0 - p11 - p22)
    pr = np.array([pi1, 1.0 - pi1])
    num = np.zeros(2); den = np.zeros(2)
    for t in range(n):
        cc = cue[t]
        ma = 0.0
        if use_ck and den[cc] > 0:
            ma = num[cc] / den[cc]
        pl = 0.0
        pk = np.zeros(2)
        for k in range(2):
            z = P[2 + k] + P[4 + k] * cue[t] + P[6] * sat[t]
            if use_ck:
                z += P[8] * ma
            if z > 30.0: z = 30.0
            elif z < -30.0: z = -30.0
            pk[k] = 1.0 / (1.0 + math.exp(-z))
            pl += pr[k] * pk[k]
        a = action[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12: pa = 1e-12
        ll[t] = math.log(pa)
        po0 = pr[0] * (pk[0] if a == 1 else 1.0 - pk[0])
        po1 = pr[1] * (pk[1] if a == 1 else 1.0 - pk[1])
        sm = po0 + po1
        if sm < 1e-300: sm = 1e-300
        po0 /= sm; po1 /= sm
        pr[0] = po0 * p11 + po1 * (1.0 - p22)
        pr[1] = po0 * (1.0 - p11) + po1 * p22
        if use_ck:
            rho = P[7]
            num[cc] = rho * num[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0


def loglik(model, params, action, reward, sat, cue, state=None, traces=False):
    n = len(action)
    if model in ("hmm", "hmmck"):
        ll = np.empty(n)
        _hmm(np.asarray(params, float), np.array(action, dtype=np.int64), np.asarray(sat, float),
             np.asarray(cue, np.int64), model == "hmmck", ll)
        return ll
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), np.array(action, dtype=np.int64),
          np.array(reward, dtype=np.int64), np.asarray(sat, float), np.zeros(n),
          np.zeros(n, np.int64), np.asarray(cue, np.int64), state, False, ll, b, d, g)
    return (ll, dict(beta=b, delta=d, g=g)) if traces else ll


def simulate(model, params, rewarded, rng, cue=None, state=None):
    n = len(rewarded)
    cue = rewarded if cue is None else cue
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    act = np.zeros(n, np.int64); rew = np.zeros(n, np.int64); sat = np.zeros(n)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), act, rew, sat, rng.random(n),
          np.asarray(rewarded, np.int64), np.asarray(cue, np.int64), state, True, ll, b, d, g)
    return act, rew, sat


In [ ]:
%%writefile fit.py
"""Fitting and leave-one-session-out cross-validation (per mouse)."""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize

import models as M


def nll(params, model, sessions):
    tot = 0.0
    for act, rew, sat, cue, st in sessions:
        tot -= M.loglik(model, params, act, rew, sat, cue, st).sum()
    return tot


def fit(model, sessions, extra_starts=0, rng=None):
    names, bounds, starts = M.SPECS[model]
    starts = [np.array(s, float) for s in starts]
    if extra_starts:
        rng = np.random.default_rng(0) if rng is None else rng
        lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
        for _ in range(extra_starts):
            starts.append(lo + rng.random(len(lo)) * (hi - lo))
    best = None
    for s0 in starts:
        r = minimize(nll, s0, args=(model, sessions), method="L-BFGS-B", bounds=bounds)
        if best is None or r.fun < best.fun:
            best = r
    return best.x, best.fun


def loso(mouse_sessions, models_list=None, extra_starts=0):
    """mouse_sessions: list of (act, rew, sat) for ONE mouse. Returns dict model -> array of
    held-out mean log-lik per trial for each session (and the fitted params per fold)."""
    models_list = models_list or M.MODELS
    k = len(mouse_sessions)
    out = {m: np.full(k, np.nan) for m in models_list}
    par = {m: [None] * k for m in models_list}
    if k < 2:
        return out, par
    for i in range(k):
        train = [s for j, s in enumerate(mouse_sessions) if j != i]
        te = mouse_sessions[i]
        for m in models_list:
            p, _ = fit(m, train, extra_starts)
            out[m][i] = M.loglik(m, p, *te).mean()
            par[m][i] = p
    return out, par


def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x))).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


In [ ]:
# Sube data_train.zip, velocidad.zip, data_pupil.zip
import zipfile
from google.colab import files
up=files.upload()
for k in up:
    if k.endswith('.zip'): zipfile.ZipFile(k).extractall('.')
!ls

In [ ]:
%%writefile run_running.py
"""Real-data test: does the DMA gate trace g_t (published constants, from observed licks/rewards) predict trial-level running speed
(arousal proxy) beyond |delta| and nuisance terms?  Pre-specified: outcome z(log1p(run_pre)) over kept trials; predictors z(g_t),
z(|delta_{t-1}|), z(satiety), z(trial index) fitted within session; mouse-level mean slope of g_t with bootstrap CI;
null = circular shift of g_t within session (keeps its autocorrelation).  Usage: python run_running.py DATA OUT.json [prefix=run] [column=run_pre]   (pupil: prefix=pupil column=pupil_pre)"""
import sys, os, json
import numpy as np, pandas as pd
import models as M, fit as F, run_real as R

D, OUT = sys.argv[1], sys.argv[2]
PREFIX = sys.argv[3] if len(sys.argv) > 3 else "run"; COL = sys.argv[4] if len(sys.argv) > 4 else "run_pre"
rng = np.random.default_rng(0)
sess = pd.read_csv(os.path.join(D, "sessions.csv")); P = np.array([-5.0, 0.0, 3.0])
z = lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0
per = {}
for _, r in sess.iterrows():
    tp = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"]); rp = os.path.join(D, "%s_%d.csv" % (PREFIX, r["behavior_session_id"]))
    if not (os.path.exists(tp) and os.path.exists(rp)): continue
    t = pd.read_csv(tp); run = pd.read_csv(rp)
    if len(run) != len(t): continue
    for c in ["aborted", "auto_rewarded"]:
        if c not in t.columns: t[c] = False
    t["run_pre"] = run[COL].to_numpy()
    t = t[(~t["aborted"].astype(bool)) & (~t["auto_rewarded"].astype(bool))]
    t = t[t["go"].astype(bool) | t["catch"].astype(bool)].reset_index(drop=True)
    lick = (t["hit"].astype(bool) | t["false_alarm"].astype(bool)).astype(int).to_numpy(); rew = t["hit"].astype(bool).astype(int).to_numpy()
    cue = t["go"].astype(bool).astype(int).to_numpy(); sat = np.concatenate([[0], np.cumsum(rew)[:-1]]) / 100.0
    if len(lick) < 150: continue
    _, tr = M.loglik("dma_const", P, lick, rew, sat, cue, traces=True)
    y = np.log1p(np.clip(t["run_pre"].to_numpy(float), 0, None)); ok = np.isfinite(y)
    if ok.sum() < 100: continue
    dprev = np.concatenate([[0], np.abs(tr["delta"])[:-1]])
    per.setdefault(r["mouse_id"], []).append((y, tr["g"], dprev, sat, np.arange(len(y)), ok))

def slope(y, g, dprev, sat, idx, ok):
    X = np.column_stack([np.ones(ok.sum()), z(g[ok]), z(dprev[ok]), z(sat[ok]), z(idx[ok])])
    return np.linalg.lstsq(X, z(y[ok]), rcond=None)[0][1]

mice = {m: L for m, L in per.items() if len(L) >= 2}
obs = np.array([np.mean([slope(*s) for s in L]) for L in mice.values()])
reps = []
for _ in range(500):
    v = []
    for L in mice.values():
        w = []
        for y, g, dp, sat, idx, ok in L:
            k = int(rng.integers(20, len(g) - 20)); w.append(slope(y, np.roll(g, k), dp, sat, idx, ok))
        v.append(np.mean(w))
    reps.append(np.mean(v))
reps = np.array(reps); mean = obs.mean(); ci = F.boot_ci(obs)
out = dict(n_mice=len(mice), n_sessions=sum(len(L) for L in mice.values()), mean_slope=float(mean), ci=list(map(float, ci)),
           p_shift=float((np.abs(reps) >= abs(mean)).mean()), frac_positive=float((obs > 0).mean()), per_mouse=obs.tolist())
json.dump(out, open(OUT, "w"), indent=1); print({k: v for k, v in out.items() if k != "per_mouse"})


In [ ]:
%%writefile run_running_controls.py
"""Secondary (post hoc, labelled as such) robustness of the g_t -> arousal result.
g_t is a smoothed unsigned prediction error, so it can track slow engagement or reward history rather than the beta gate.
Slope of z(g_t) on z(log1p(arousal)) with (A) baseline nuisance terms [|delta_{t-1}|, satiety, trial index] and
(B) baseline + matched slow traces with the gate's own time constant (tau = 45): EMA of licks, EMA of rewards, EMA of |delta|.
Also: slope of EMA|delta| alone (A-style) for comparison.  Null: circular shift of g_t within session.
Usage: python run_running_controls.py DATA OUT.json PREFIX COLUMN"""
import sys, os, json
import numpy as np, pandas as pd
import models as M, fit as F

D, OUT, PREFIX, COL = sys.argv[1:5]; rng = np.random.default_rng(0)
sess = pd.read_csv(os.path.join(D, "sessions.csv")); P = np.array([-5.0, 0.0, 3.0]); TAU = 45.0
z = lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0
def ema_before(x):  # value of the EMA available BEFORE trial t
    out = np.zeros(len(x)); m = 0.0
    for t in range(len(x)):
        out[t] = m; m += (x[t] - m) / TAU
    return out
per = {}
for _, r in sess.iterrows():
    tp = os.path.join(D, "trials_%d.csv" % r["behavior_session_id"]); rp = os.path.join(D, "%s_%d.csv" % (PREFIX, r["behavior_session_id"]))
    if not (os.path.exists(tp) and os.path.exists(rp)): continue
    t = pd.read_csv(tp); run = pd.read_csv(rp)
    if len(run) != len(t): continue
    for c in ["aborted", "auto_rewarded"]:
        if c not in t.columns: t[c] = False
    t["y"] = run[COL].to_numpy()
    t = t[(~t["aborted"].astype(bool)) & (~t["auto_rewarded"].astype(bool))]
    t = t[t["go"].astype(bool) | t["catch"].astype(bool)].reset_index(drop=True)
    lick = (t["hit"].astype(bool) | t["false_alarm"].astype(bool)).astype(int).to_numpy(); rew = t["hit"].astype(bool).astype(int).to_numpy()
    cue = t["go"].astype(bool).astype(int).to_numpy(); sat = np.concatenate([[0], np.cumsum(rew)[:-1]]) / 100.0
    if len(lick) < 150: continue
    _, tr = M.loglik("dma_const", P, lick, rew, sat, cue, traces=True)
    y = np.log1p(np.clip(t["y"].to_numpy(float), 0, None)); ok = np.isfinite(y)
    if ok.sum() < 100: continue
    ad = np.abs(tr["delta"]); dprev = np.concatenate([[0], ad[:-1]])
    ctrl = np.column_stack([ema_before(lick.astype(float)), ema_before(rew.astype(float)), ema_before(ad)])
    per.setdefault(r["mouse_id"], []).append(dict(y=y, g=tr["g"], dprev=dprev, sat=sat, idx=np.arange(len(y)), ok=ok, ctrl=ctrl, emad=ctrl[:, 2]))

def slope(s, g, extended):
    k = s["ok"]; cols = [np.ones(k.sum()), z(g[k]), z(s["dprev"][k]), z(s["sat"][k]), z(s["idx"][k])]
    if extended: cols += [z(s["ctrl"][k][:, j]) for j in range(3)]
    return np.linalg.lstsq(np.column_stack(cols), z(s["y"][k]), rcond=None)[0][1]
mice = {m: L for m, L in per.items() if len(L) >= 2}
def mm(fn): return np.array([np.mean([fn(s) for s in L]) for L in mice.values()])
res = {"n_mice": len(mice), "n_sessions": sum(len(L) for L in mice.values())}
for name, fn in [("A_g_baseline", lambda s: slope(s, s["g"], False)), ("B_g_with_slow_controls", lambda s: slope(s, s["g"], True)),
                 ("C_emaAbsDelta_baseline", lambda s: slope(s, s["emad"], False))]:
    v = mm(fn); m, lo, hi = F.boot_ci(v); res[name] = dict(slope=float(m), ci=[float(lo), float(hi)], frac_pos=float((v > 0).mean()))
reps = []
for _ in range(400):
    vals = []
    for L in mice.values():
        w = []
        for s in L:
            k = int(rng.integers(20, len(s["g"]) - 20)); w.append(slope(s, np.roll(s["g"], k), True))
        vals.append(np.mean(w))
    reps.append(np.mean(vals))
reps = np.array(reps); res["B_p_shift"] = float((np.abs(reps) >= abs(res["B_g_with_slow_controls"]["slope"])).mean())
json.dump(res, open(OUT, "w"), indent=1); print(json.dumps(res))


In [ ]:
%%writefile make_figs.py
import json, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import fit as F
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "font.size": 7.5,
                     "axes.linewidth": .6, "xtick.major.width": .6, "ytick.major.width": .6})
V = {}; [V.update(json.load(open(f))) for f in ("vol_a.json", "vol_b.json")]
tasks = ["detect", "200", "100", "50", "25"]; lab = ["stationary", "200", "100", "50", "25"]
x = np.arange(len(tasks))
fig, ax = plt.subplots(2, 1, figsize=(3.3, 3.55), gridspec_kw=dict(height_ratios=[1, 1.05]))
a = ax[0]
for truth, mk, col, off in [("dma", "o", "#1b6ca8", -.08), ("q", "s", "#777777", .08)]:
    m = np.array([V[t][truth]["dma_minus_q"][0] for t in tasks]) * 1e4
    lo = np.array([V[t][truth]["dma_minus_q"][1] for t in tasks]) * 1e4
    hi = np.array([V[t][truth]["dma_minus_q"][2] for t in tasks]) * 1e4
    a.errorbar(x + off, m, yerr=[m - lo, hi - m], fmt=mk, color=col, ms=3.5, capsize=2, lw=.8,
               label="data generated by DMA" if truth == "dma" else "data generated by TD (fixed $\\beta$)",
               mfc=col if truth == "dma" else "white")
a.axhline(0, color="k", lw=.5, ls=":")
a.set_ylim(-13, 4.5); a.set_xticks(x); a.set_xticklabels(lab); a.set_xlabel("reward-contingency block length (trials)")
a.set_ylabel("DMA $-$ TD held-out LL\n($10^{-4}$ nats/trial)"); a.legend(frameon=False, fontsize=6.3, loc="lower left")
a.set_title("(a) Simulated mice (60 per cell)", fontsize=7.5, loc="left")
R = json.load(open("real30_primary.json")); E = json.load(open("real30_engaged.json"))
b = ax[1]
groups = [("DMA$-$TD\nall sessions", R, "dma", "#1b6ca8"), ("DMA$-$TD\nengaged", E, "dma", "#1b6ca8"), ("PH$-$TD\nall sessions", R, "ph", "#b5651d")]
rng = np.random.default_rng(1)
for i, (nm, J, mod, col) in enumerate(groups):
    d = (np.array(J["per_mouse_ll"][mod]) - np.array(J["per_mouse_ll"]["q"])) * 1e4
    b.scatter(i + rng.uniform(-.12, .12, len(d)), d, s=9, color=col, alpha=.75, lw=0, zorder=3)
    m, lo, hi = F.boot_ci(d / 1e4); b.errorbar(i + .27, m * 1e4, yerr=[[m * 1e4 - lo * 1e4], [hi * 1e4 - m * 1e4]], fmt="D", color="k", ms=3, capsize=2, lw=.8, zorder=4)
b.axhline(0, color="k", lw=.5, ls=":"); b.set_xticks(range(3)); b.set_xticklabels([g[0] for g in groups])
b.set_ylabel("held-out LL vs TD\n($10^{-4}$ nats/trial, symlog)"); b.set_title("(b) Allen mice: animals (dots), mean and 95% CI (diamonds)", fontsize=7.3, loc="left")
b.set_yscale("symlog", linthresh=10); b.set_ylim(-150, 600); b.set_yticks([-100, -10, 0, 10, 100, 500]); b.set_yticklabels(["$-$100", "$-$10", "0", "10", "100", "500"])
plt.tight_layout(pad=.4, h_pad=.9); fig.savefig("fig1.png", dpi=300)
print("fig ok")
# bootstrap CI vs TD for each model + counts for tables
for tag, J in (("primary", R), ("engaged", E)):
    print(tag, {m: round(np.mean(J["per_mouse_ll"][m]), 4) for m in J["per_mouse_ll"]})
    for m in ["bias", "ph", "dma", "dma_const"]:
        d = np.array(J["per_mouse_ll"][m]) - np.array(J["per_mouse_ll"]["q"]); mm, lo, hi = F.boot_ci(d)
        print("   ", m, "- q: %.1f [%.1f, %.1f]" % (mm * 1e4, lo * 1e4, hi * 1e4))
print("SIM TABLE")
for t, l in zip(tasks, lab):
    d, q = V[t]["dma"], V[t]["q"]; P = d["param_recovery"]
    r = lambda i: np.corrcoef([p["true"][i] for p in P], [p["est"][i] for p in P])[0, 1]
    print(l, "| dma-q (truth dma) %.1f [%.1f,%.1f]" % tuple(np.array(d["dma_minus_q"]) * 1e4), "| (truth q) %.1f [%.1f,%.1f]" % tuple(np.array(q["dma_minus_q"]) * 1e4),
          "| r_lam %.2f r_tau %.2f" % (r(2), r(3)), "| dma wins %d/60" % d["wins"]["dma"])


In [ ]:
%%writefile make_fig2.py
import json, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.linewidth": .6})
V = {}; [V.update(json.load(open(f))) for f in ("vol_a.json", "vol_b.json")]
fig, ax = plt.subplots(1, 2, figsize=(3.3, 1.75))
for a, idx, nm, lim in [(ax[0], 2, r"gate strength $\lambda$", (0, 1.5)), (ax[1], 3, r"gate time constant $\tau$ (trials)", (0, 300))]:
    for t, mk, col, lab in [("detect", "o", "#1b6ca8", "stationary"), ("25", "s", "#b5651d", "$L$ = 25")]:
        P = V[t]["dma"]["param_recovery"]
        x = [p["true"][idx] for p in P]; y = [p["est"][idx] for p in P]
        r = np.corrcoef(x, y)[0, 1]
        a.scatter(x, y, s=6, marker=mk, color=col, alpha=.7, lw=0, label="%s, r=%.2f" % (lab, r))
    lo, hi = (0.4, 1.3) if idx == 2 else (5, 85)
    a.plot([lo, hi], [lo, hi], "k:", lw=.6); a.set_ylim(*lim)
    a.set_xlabel("true " + nm.split(" ")[0] + " " + nm.split(" ")[1] + (" " + nm.split(" ")[2] if len(nm.split(" ")) > 2 else ""), fontsize=6.5)
    a.set_ylabel("estimated", fontsize=6.5)
ax[0].set_title(r"(a) $\lambda$: $r$ = 0.33, 0.24", fontsize=6.8, loc="left"); ax[1].set_title(r"(b) $\tau$: $r$ = 0.31, 0.20", fontsize=6.8, loc="left")
plt.tight_layout(pad=.4, w_pad=.8); fig.savefig("fig2.png", dpi=300); print("ok")


## Velocidad de carrera (25 ratones, sesiones de entrenamiento)

In [ ]:
!python run_running.py data_train run_train.json run run_pre
!python run_running_controls.py data_train run_train_ctrl.json run run_pre

## Pupila y velocidad en sesiones de imagen (mismos ratones)

In [ ]:
!python run_running.py data_pupil pupil2.json pupil pupil_pre
!python run_running.py data_pupil run_ophys2.json pupil run_pre
!python run_running_controls.py data_pupil pupil2_ctrl.json pupil pupil_pre
!python run_running_controls.py data_pupil run_ophys2_ctrl.json pupil run_pre

## Figuras (necesitan los .json de las notebooks 2 y 3 en esta carpeta)

In [ ]:
!python make_figs.py
!python make_fig2.py